<a href="https://colab.research.google.com/github/ShahidNauman/ist-assignments/blob/feature%2Fmini-transformer/transformer_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install -q pypdf transformers sentencepiece

In [23]:
from google.colab import files
import os

# Create data folder
os.makedirs("data", exist_ok=True)

# Ask to upload the research work.
uploaded = files.upload()

# Move uploaded PDFs into data/
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")

print("Files moved to data/ folder.")

Saving 20190116.pdf to 20190116.pdf
Saving 20190923.pdf to 20190923.pdf
Saving 20200804.pdf to 20200804.pdf
Files moved to data/ folder.


In [24]:
import json
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

from pypdf import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

DATA_DIR = Path("data")
OUTPUT_FILE = "agenda_summary.json"
MODEL_NAME = "facebook/bart-large-cnn" # "google/flan-t5-small"
LOCAL_CHUNK_SIZE = 250
STOPWORDS = {
    "a","an","and","are","as","at","be","by","for","from","has","he","in","is","it","its",
    "of","on","that","the","to","was","were","will","with","we","our","this","these",
    "those","their","they","you","your","or","not","can","may","using","use","used",
    "also","such","than","into","over","via","based","new","approach","paper","study",
    "method","methods","results","research",
}

In [25]:
@dataclass
class Paper:
    path: Path
    raw_text: str


def read_pdf_text(pdf_path):
    reader = PdfReader(str(pdf_path))
    return "\n".join([page.extract_text() or "" for page in reader.pages])


def clean_text(text):
    text = text.replace("\x00", " ")
    return re.sub(r"\s+", " ", text).strip()


def load_papers(data_dir):
    pdfs = sorted(data_dir.glob("*.pdf"))
    papers = []
    for pdf in pdfs:
        papers.append(
            Paper(
                path=pdf,
                raw_text=clean_text(read_pdf_text(pdf)),
            )
        )
    return papers


def chunk_words(text, size):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

In [26]:
def build_summarizer(model_name):
    try:
        summarizer = pipeline("summarization", model=model_name)

        def fn(text, max_length=140, min_length=40):
            return summarizer(
                text,
                max_length=max_length,
                min_length=min_length,
                do_sample=False,
                truncation=True
            )[0]["summary_text"]
        return fn

    except:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        def fn(text, max_length=140, min_length=40):
            input_text = f"summarize: {text}"
            inputs = tokenizer(input_text, return_tensors="pt", truncation=True)
            outputs = model.generate(**inputs, max_length=max_length, min_length=min_length)
            return tokenizer.decode(outputs[0], skip_special_tokens=True)

        return fn

In [ ]:
print("Loading papers...")
papers = load_papers(DATA_DIR)

print("Loading model...")
summarizer = build_summarizer(MODEL_NAME)

per_paper = []

print("Summarizing papers...")
for paper in papers:
    chunks = chunk_words(paper.raw_text, LOCAL_CHUNK_SIZE)
    summaries = summarize_chunks(chunks, summarizer)

    merged = " ".join(summaries)
    second_pass = summarize_chunks(chunk_words(merged, LOCAL_CHUNK_SIZE), summarizer)

    final_summary = " ".join(second_pass) if second_pass else merged

    per_paper.append({
        "file": paper.path.name,
        "summary": final_summary,
    })

print("Building global summary...")
all_text = " ".join(p.raw_text for p in papers)
global_chunks = chunk_words(all_text, LOCAL_CHUNK_SIZE)
global_summaries = summarize_chunks(global_chunks, summarizer)

final_global = summarize_chunks(
    chunk_words(" ".join(global_summaries), LOCAL_CHUNK_SIZE),
    summarizer,
    max_length=200,
    min_length=70,
)

global_summary = " ".join(final_global)

themes = extract_top_terms(
    global_summary + " " + " ".join(p["summary"] for p in per_paper)
)

evolution = build_evolution_summary(per_paper)

# Filter per_paper to only include 'file' and 'summary' for the final export
per_paper_output = [
    {"file": p["file"], "summary": p["summary"]}
    for p in per_paper
]

result = {
    "global_research_agenda_summary": global_summary,
    "key_themes_and_recurring_ideas": themes,
    "evolution_over_time": evolution,
    "per_paper_summaries": per_paper_output,
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(result, f, indent=2)

print("Done! Saved to:", OUTPUT_FILE)

Loading papers...
Loading model...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summarizing papers...


In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)